# 원본 vs 평면화 이미지 YOLO 탐지 비교

`homography_data/`(초록 테이프만 붙인 원본 촬영 이미지)와 `homography_result/`(A4 평면화 결과)를
같은 `best_v8.pt` 모델로 각각 YOLO 탐지하여, 평면화가 타일 탐지 정확도에 어떤 영향을 주는지 나란히 비교합니다.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

BASE_DIR = Path.cwd()
MODEL_PATH = BASE_DIR / "best_v8.pt"
RAW_DIR = BASE_DIR / "homography_data"
FLAT_DIR = BASE_DIR / "homography_result"
CONFIDENCE = 0.25

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

In [ ]:
import matplotlib.font_manager as fm

# matplotlib 기본 폰트는 한글을 지원하지 않아 그래프 제목이 네모(□)로 깨져 보입니다.
# 1) 시스템에 이미 설치된 한글 폰트를 먼저 찾고,
# 2) 없으면 노트북 옆의 `fonts/` 폴더에 놓아둔 .ttf 파일을 찾아 등록합니다.
#    (인터넷이 안 되는 폐쇄망 서버라면, 로컬 PC의 C:\Windows\Fonts\malgun.ttf 파일을
#     이 프로젝트의 `fonts/` 폴더로 복사해 넣고 이 셀을 다시 실행하세요.)
_installed_fonts = {f.name for f in fm.fontManager.ttflist}
_font_name = next(
    (n for n in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]
     if n in _installed_fonts),
    None,
)

if _font_name is None:
    _font_dir = BASE_DIR / "fonts"
    _local_fonts = sorted(_font_dir.glob("*.ttf")) if _font_dir.exists() else []
    if _local_fonts:
        fm.fontManager.addfont(str(_local_fonts[0]))
        _font_name = fm.FontProperties(fname=str(_local_fonts[0])).get_name()
        print(f"로컬 폰트 등록: {_local_fonts[0].name} -> {_font_name}")
    else:
        print(
            "한글 폰트를 찾지 못했습니다. 그래프 제목이 깨질 수 있습니다.\n"
            f"해결하려면 한글 .ttf 폰트 파일을 '{_font_dir}' 폴더에 넣고 이 셀을 다시 실행하세요."
        )

if _font_name:
    plt.rcParams["font.family"] = _font_name

plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트 사용 시 마이너스(-) 기호 깨짐 방지

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"모델 파일이 없습니다: {MODEL_PATH.resolve()}")
if not RAW_DIR.exists():
    raise FileNotFoundError(f"원본 이미지 폴더가 없습니다: {RAW_DIR.resolve()}")
if not FLAT_DIR.exists():
    raise FileNotFoundError(f"평면화 이미지 폴더가 없습니다: {FLAT_DIR.resolve()}")

device = 0 if torch.cuda.is_available() else "cpu"
print("사용 장치:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO(str(MODEL_PATH))

In [ ]:
raw_paths = sorted(
    p for p in RAW_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

pairs = []
for raw_path in raw_paths:
    flat_candidates = sorted(FLAT_DIR.glob(f"{raw_path.stem}_flat.*"))
    if not flat_candidates:
        print(f"[경고] {raw_path.name}에 대응하는 평면화 이미지를 찾지 못했습니다 (건너뜀).")
        continue
    pairs.append((raw_path, flat_candidates[0]))

print(f"비교할 이미지 쌍: {len(pairs)}개")
for raw_path, flat_path in pairs:
    print(f" - {raw_path.name}  <->  {flat_path.name}")

In [ ]:
def _print_detections(label: str, result) -> int:
    boxes = result.boxes
    count = len(boxes) if boxes is not None else 0
    if count > 0:
        print(f"  [{label}] 탐지 {count}개:")
        for cls_id, conf in zip(boxes.cls.tolist(), boxes.conf.tolist()):
            print(f"    - {result.names[int(cls_id)]}: {conf:.2f}")
    else:
        print(f"  [{label}] 탐지된 객체 없음")
    return count


summary = []

for raw_path, flat_path in pairs:
    raw_result = model.predict(source=str(raw_path), conf=CONFIDENCE, device=device, verbose=False)[0]
    flat_result = model.predict(source=str(flat_path), conf=CONFIDENCE, device=device, verbose=False)[0]

    raw_annotated = cv2.cvtColor(raw_result.plot(), cv2.COLOR_BGR2RGB)
    flat_annotated = cv2.cvtColor(flat_result.plot(), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(raw_annotated)
    axes[0].set_title(f"원본 (homography_data)\n{raw_path.name}")
    axes[0].axis("off")

    axes[1].imshow(flat_annotated)
    axes[1].set_title(f"평면화 (homography_result)\n{flat_path.name}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    print(f"[{raw_path.stem}]")
    raw_count = _print_detections("원본", raw_result)
    flat_count = _print_detections("평면화", flat_result)
    print()

    summary.append((raw_path.stem, raw_count, flat_count))

In [ ]:
print(f"{'파일':40s} {'원본 탐지수':>10s} {'평면화 탐지수':>12s}")
for stem, raw_count, flat_count in summary:
    print(f"{stem:40s} {raw_count:>10d} {flat_count:>12d}")

## homography한 데이터 대상으로 학습요함